# Devoir 1: Pipeline NLP pour la structuration du Code de la Route Marocain

## Objectif: Transformer un texte arabe brut en CSV structuré avec les entités juridiques

**Colonnes cibles:** article_id, infraction_desc, categorie_vehicule, amende_fixe, points_retrait, mots_cles

In [ ]:
# ============================================
# ÉTAPE 1: Installation des dépendances
# ============================================

!pip install pyarabic pandas spacy -q
!python -m spacy download ar_core_news_sm -q

import re
import pandas as pd
import numpy as np
from pyarabic.araby import strip_tashkeel, strip_tatweel, normalize_hamza, normalize_lamalef
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

## Étape A: Prétraitement Arabe (Normalisation)

In [ ]:
class ArabicNormalizer:
    """Normalisation du texte arabe: Hamza, Ta Marbuta, Tashkeel"""
    
    def __init__(self):
        # Table de correspondance pour les caractères arabes
        self.hamza_map = {
            'أ': 'ا', 'إ': 'ا', 'آ': 'ا',  # Hamza normalisé en Alif
            'ؤ': 'و',  # Waw avec hamza
            'ئ': 'ي',  # Ya avec hamza
        }
        # Ta Marbuta (ة) -> Ha (ه) ou simple?
        self.ta_marbuta = 'ة'
        self.ta_mabsuta = 'ه'
        
    def normalize(self, text):
        """Normalisation complète du texte arabe"""
        if not isinstance(text, str):
            return ""
        
        # 1. Suppression des voyelles (Tashkeel)
        text = strip_tashkeel(text)
        
        # 2. Suppression des Tatweel (ـ)
        text = strip_tatweel(text)
        
        # 3. Normalisation des Hamzas
        text = normalize_hamza(text)
        
        # 4. Normalisation Lam-Alef
        text = normalize_lamalef(text)
        
        # 5. Conversion Ta Marbuta (optionnel, garder pour détection)
        # text = text.replace(self.ta_marbuta, self.ta_mabsuta)
        
        # 6. Conversion des chiffres arabes en chiffres latins
        arabic_numbers = {
            '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4',
            '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9'
        }
        for ar, lat in arabic_numbers.items():
            text = text.replace(ar, lat)
        
        # 7. Suppression des espaces multiples
        text = re.sub(r'\s+', ' ', text)
        
        return text.strip()


normalizer = ArabicNormalizer()

# Test
test = "السَّلامُ عَلَيْكُمْ وَرَحْمَةُ اللَّهِ"
print(f"Original: {test}")
print(f"Normalisé: {normalizer.normalize(test)}")

## Étape B: Segmentation (Tokenization) et Extraction par Patterns (Rules-based NLP)

In [ ]:
class LegalPatternExtractor:
    """Extraction des entités juridiques par patterns regex"""
    
    def __init__(self):
        # Pattern pour extraire les amendes
        self.amende_patterns = [
            # غرامة من X إلى Y درهم
            re.compile(r'غرامة\s+من\s+(\d+(?:[.,]\d+)?)\s*(?:إلى|–|-)\s*(\d+(?:[.,]\d+)?)\s*درهم', re.UNICODE),
            # غرامة قدرها X درهم
            re.compile(r'غرامة\s+(?:قدرها|تبلغ)\s+(\d+(?:[.,]\d+)?)\s*درهم', re.UNICODE),
            # X إلى Y درهم
            re.compile(r'(\d+(?:[.,]\d+)?)\s*(?:إلى|–|-)\s*(\d+(?:[.,]\d+)?)\s*درهم', re.UNICODE),
            # simple: X درهم
            re.compile(r'(\d+(?:[.,]\d+)?)\s*درهم', re.UNICODE),
        ]
        
        # Pattern pour extraire les points à retirer
        self.points_patterns = [
            re.compile(r'خصم\s+(\d+)\s+نقط', re.UNICODE),
            re.compile(r'(\d+)\s+(?:نقطة|نقاط|نقط)', re.UNICODE),
            re.compile(r'النقط\s+الواجب\s+خصمها\s*(?::|\|)?\s*(\d+)', re.UNICODE),
        ]
        
        # Pattern pour les catégories de véhicules
        self.categorie_patterns = {
            'poids_lourd': re.compile(r'(شاحنة|نقل\s+البضائع|وزنها\s+الإجمالي|PTC|حمولة|شاحنات|حافلة|نقل\s+عمومي)', re.UNICODE),
            'moto': re.compile(r'(دراجة|دراجات|نارية|سكوتر|محرك|عجلتين)', re.UNICODE),
            'vehicule_agricole': re.compile(r'(فلاحية|غابوية|جرار|أشغال\s+عمومية)', re.UNICODE),
            'vehicule_remorque': re.compile(r'(مقطورة|جرار|مركبة\s+مقرونة)', re.UNICODE),
        }
        
        # Pattern pour extraire les mots-clés (infractions)
        self.keyword_patterns = {
            'vitesse': re.compile(r'(سرعة|تجاوز|السرعة|كلم\s*في\s*الساعة)', re.UNICODE),
            'stationnement': re.compile(r'(توقف|وقوف|الوقوف|التوقف|مرآب|ركن)', re.UNICODE),
            'alcohol': re.compile(r'(كحول|سكر|مسكر|خمر)', re.UNICODE),
            'drogue': re.compile(r'(مخدر|مؤثرات|عقلية|منشطات)', re.UNICODE),
            'nuit': re.compile(r'(ليل|ليلاً|الليل|مساء)', re.UNICODE),
            'autoroute': re.compile(r'(طريق\s+سيار|الطرق\s+السيارة|مسلك|الطريق\s+السيار)', re.UNICODE),
            'ceinture': re.compile(r'(حزام|السلامة|حزام\s+السلامة)', re.UNICODE),
            'telephone': re.compile(r'(هاتف|جوال|محمول|الاتصال)', re.UNICODE),
            'depassement': re.compile(r'(تجاوز|تخطي|تجاوز\s+غير\s+قانوني)', re.UNICODE),
            'feux': re.compile(r'(إضاءة|أضواء|أنوار|تشوير|إشارة)', re.UNICODE),
        }
        
        # Pattern pour détecter le type de texte
        self.type_patterns = {
            'definition': re.compile(r'(يقصد|يعتبر|يراد|هو كل|هي كل|المقصود)', re.UNICODE),
            'obligation': re.compile(r'(يجب على|يلتزم|يتعين|على كل|من واجب)', re.UNICODE),
            'interdiction': re.compile(r'(يمنع|لا يجوز|ممنوع|يحظر|لا يحق)', re.UNICODE),
            'sanction': re.compile(r'(يعاقب|غرامة|الحبس|تسحب رخصة|توقف رخصة|خصم|نقط)', re.UNICODE),
        }
    
    def extract_amendes(self, text):
        """Extrait les montants d'amende"""
        text = normalizer.normalize(text)
        for pattern in self.amende_patterns:
            match = pattern.search(text)
            if match:
                if len(match.groups()) == 2:
                    # Range: min to max
                    min_val = float(match.group(1).replace(',', '.'))
                    max_val = float(match.group(2).replace(',', '.'))
                    return (min_val, max_val)
                elif len(match.groups()) == 1:
                    val = float(match.group(1).replace(',', '.'))
                    return (val, val)
        return None
    
    def extract_points(self, text):
        """Extrait le nombre de points à retirer"""
        text = normalizer.normalize(text)
        for pattern in self.points_patterns:
            match = pattern.search(text)
            if match:
                return int(match.group(1))
        return None
    
    def extract_categorie(self, text):
        """Détecte la catégorie de véhicule concernée"""
        text = normalizer.normalize(text)
        categories = []
        for cat, pattern in self.categorie_patterns.items():
            if pattern.search(text):
                categories.append(cat)
        return categories if categories else ['tous']
    
    def extract_keywords(self, text):
        """Extrait les mots-clés décrivant l'infraction"""
        text = normalizer.normalize(text)
        keywords = []
        for kw, pattern in self.keyword_patterns.items():
            if pattern.search(text):
                keywords.append(kw)
        return keywords
    
    def detect_type(self, text):
        """Détecte le type de paragraphe"""
        text = normalizer.normalize(text)
        for typ, pattern in self.type_patterns.items():
            if pattern.search(text):
                return typ
        return 'information'
    
    def extract_article_id(self, text):
        """Extrait le numéro d'article"""
        pattern = re.compile(r'المادة\s+(\d+(?:[\s\-]+\d+)?)', re.UNICODE)
        match = pattern.search(text)
        if match:
            return match.group(1).strip().replace(' ', '-')
        return None
    
    def extract_infraction_desc(self, text, article_id=None):
        """Extrait la description de l'infraction"""
        # Nettoyer le texte
        desc = normalizer.normalize(text)
        # Supprimer l'en-tête d'article
        desc = re.sub(r'المادة\s+\d+(?:[\s\-]+\d+)?\s*[:.]?\s*', '', desc)
        # Supprimer les références "أعاله", "أسفله"
        desc = re.sub(r'\s*[أاإ][^\s]*ه\s*', ' ', desc)
        # Nettoyer les espaces
        desc = re.sub(r'\s+', ' ', desc).strip()
        return desc


extractor = LegalPatternExtractor()

## Chargement et parsing du texte du Code de la Route

In [ ]:
# Données d'exemple du Code de la Route (extraites du PDF)

raw_text = """
المادة 184
يعاقب كل شخص ارتكب مخالفة من الدرجة الأولى بغرامة من سبعمائة (700) إلى ألف وأربعمائة (1.400) درهم.
: تعتبر مخالفة من الدرجة الأولى إحدى المخالفات التالية
1 - تجاوز السرعة القصوى المسموح بها بثلاثين (30) إلى أقل من خمسين (50) كيلومترا في الساعة، بالنسبة لجميع السائقين؛
2 - سير مركبة على الطريق العمومية، خارج التجمعات العمرانية، ليلا دون إنارة؛
3 - التوقف المخالف للنصوص الجاري بها العمل، ليلا من غير أضواء، خارج التجمعات العمرانية؛
4 - عدم احترام الوقوف المفروض بعلامة قف أو بضوء التشوير الأحمر؛
5 - التوقف الخطير لمركبة، عندما تكون الرؤية غير كافية، بالقرب من منعرج أو بالقرب من قمة منحدر أو على قنطرة أو داخل نفق أو التوقف الذي يحجب التشوير أو التوقف على بعد أقل من عشرة (10) أمتار من تقاطع للطرق؛
6 - قطع خط متصل؛
7 - وقوف مركبة على القناطر أو تحتها أو داخل الأنفاق أو الممرات تحت الأرضية أو على ممر علوي، ما عدا في حالة قوة قاهرة؛
8 - التجاوز المعيب؛
9 - وقوف أو توقف مركبة على مستوى تقاطع طريق مع سكة حديدية أو بالقرب منه؛
10 - السير في اتجاه ممنوع؛

المادة 99
يخصم من رصيد النقط المخصص لرخصة السياقة:
الرقم الترتيبي | المخالفة | النقط الواجب خصمها
1 | القتل غير العمدي مع ظروف التشديد، إثر حادثة سير (ما لم يتقرر إلغاء رخصة السياقة) | 14
2 | القتل غير العمدي بدون ظروف التشديد، إثر حادثة سير | 6
3 | الجروح غير العمدية المؤدية إلى عاهة دائمة مع ظروف التشديد، إثر حادثة سير | 10
4 | الجروح غير العمدية المؤدية إلى عاهة دائمة بدون ظروف التشديد، إثر حادثة سير | 4
7 | سياقة مركبة تحت تأثير الكحول أو تحت تأثير المواد المخدرة | 8
19 | عدم احترام سائق مركبة للوقوف المفروض بعلامة قف أو بإشارة الضوء الأحمر | 4
20 | تجاوز السرعة المسموح بها بما يفوق 30 كيلومترا في الساعة ويقل عن 50 كيلومترا في الساعة | 4
21 | السير في الاتجاه الممنوع | 4
22 | عدم احترام حق الأسبقية | 2
23 | التجاوز غير القانوني | 4
32 | إركاب طفل تقل سنه عن عشر سنوات بالمقاعد الأمامية للمركبة | 1

المادة 148
يعاقب بغرامة من ألف ومائتين (1.200) إلى أربعة آلاف (4.000) درهم:
1 - كل شخص يقوم عمدا بإصدار أوامر أو بارتكاب أعمال ساهمت في إحداث إحدى الوضعيات؛
2 - في حالة العود، يعاقب الفاعل بالحبس من شهر إلى ثلاثة أشهر وبضعف الغرامة؛

المادة 183
يعاقب بالحبس من ستة أشهر إلى سنة واحدة وبغرامة من خمسة آلاف (5.000) إلى عشرة آلاف (10.000) درهم أو بإحدى هاتين العقوبتين فقط، كل شخص يسوق مركبة مع وجوده في حالة سكر، أو تحت تأثير الكحول.
تأمر المحكمة بتوقيف رخصة السياقة لمدة تتراوح بين ستة أشهر وسنة واحدة.

المادة 155
يعاقب بغرامة من ألفين (2.000) إلى خمسة آلاف (5.000) درهم كل شخص استعمل رخصة السياقة الخاصة به بصفة مهنية دون أن يكون حاصلا على بطاقة سائق مهني.
"""

In [ ]:
def parse_articles_to_dataframe(text):
    """Parse le texte et génère un DataFrame structuré"""
    
    # Pattern pour trouver les articles
    article_pattern = re.compile(
        r'المادة\s+(\d+(?:[\s\-]+\d+)?)\s*[:.]?\s*(.*?)(?=(?:\nالمادة\s+\d+|\Z))',
        re.DOTALL | re.UNICODE
    )
    
    # Pattern pour trouver les infractions dans un article
    infraction_pattern = re.compile(
        r'^\s*(\d+)[\s\-–—]+\.?\s*(.+?)(?=(?:\n\s*\d+[\s\-–—]|\Z))',
        re.MULTILINE | re.DOTALL | re.UNICODE
    )
    
    rows = []
    
    for match in article_pattern.finditer(text):
        article_id = match.group(1).strip().replace(' ', '-')
        article_content = match.group(2).strip()
        
        print(f"Traitement de l'article {article_id}...")
        
        # Extraire les amendes générales de l'article
        amende_article = extractor.extract_amendes(article_content)
        
        # Chercher les infractions individuelles
        infractions = infraction_pattern.findall(article_content)
        
        if infractions:
            # Article avec liste d'infractions
            for infraction_num, infraction_desc in infractions:
                # Nettoyer la description
                infraction_desc = re.sub(r'\s+', ' ', infraction_desc).strip()
                
                row = {
                    'article_id': f"{article_id}.{infraction_num}",
                    'infraction_desc': infraction_desc,
                    'categorie_vehicule': ', '.join(extractor.extract_categorie(infraction_desc)),
                    'amende_min': amende_article[0] if amende_article else None,
                    'amende_max': amende_article[1] if amende_article else None,
                    'amende_fixe': None,
                    'points_retrait': extractor.extract_points(infraction_desc),
                    'mots_cles': ', '.join(extractor.extract_keywords(infraction_desc)),
                    'type_texte': extractor.detect_type(infraction_desc),
                    'prison_min': None,
                    'prison_max': None,
                }
                rows.append(row)
        else:
            # Article simple (une seule infraction)
            # Vérifier s'il y a un tableau dans le contenu
            table_pattern = re.compile(r'(\d+)\s*[\|]\s*(.+?)\s*[\|]\s*(\d+)', re.UNICODE)
            table_matches = table_pattern.findall(article_content)
            
            if table_matches:
                # Article sous forme de tableau
                for num, desc, points in table_matches:
                    row = {
                        'article_id': f"{article_id}.{num}",
                        'infraction_desc': desc.strip(),
                        'categorie_vehicule': ', '.join(extractor.extract_categorie(desc)),
                        'amende_min': None,
                        'amende_max': None,
                        'amende_fixe': None,
                        'points_retrait': int(points),
                        'mots_cles': ', '.join(extractor.extract_keywords(desc)),
                        'type_texte': extractor.detect_type(desc),
                        'prison_min': None,
                        'prison_max': None,
                    }
                    rows.append(row)
            else:
                # Article simple
                desc = extractor.extract_infraction_desc(article_content, article_id)
                row = {
                    'article_id': article_id,
                    'infraction_desc': desc,
                    'categorie_vehicule': ', '.join(extractor.extract_categorie(article_content)),
                    'amende_min': amende_article[0] if amende_article else None,
                    'amende_max': amende_article[1] if amende_article else None,
                    'amende_fixe': amende_article[0] if amende_article and amende_article[0] == amende_article[1] else None,
                    'points_retrait': extractor.extract_points(article_content),
                    'mots_cles': ', '.join(extractor.extract_keywords(article_content)),
                    'type_texte': extractor.detect_type(article_content),
                    'prison_min': None,
                    'prison_max': None,
                }
                rows.append(row)
    
    return pd.DataFrame(rows)


# Exécution
df = parse_articles_to_dataframe(raw_text)
print(f"\n{len(df)} infractions extraites")
df.head(15)

## Étape C: Approche par Modèle Pré-entraîné (spaCy) pour la NER

In [ ]:
# Chargement du modèle spaCy arabe
nlp = spacy.load("ar_core_news_sm")

class SpacyNERExtractor:
    """Extraction d'entités avec spaCy"""
    
    def __init__(self, nlp_model):
        self.nlp = nlp_model
        
    def extract_entities(self, text):
        """Extrait les entités nommées du texte"""
        doc = self.nlp(normalizer.normalize(text))
        entities = {
            'PER': [],   # Personnes
            'LOC': [],   # Lieux
            'ORG': [],   # Organisations
            'MISC': [],  # Divers
            'NUM': [],   # Nombres
            'DATE': [],  # Dates
        }
        for ent in doc.ents:
            if ent.label_ in entities:
                entities[ent.label_].append(ent.text)
            else:
                entities['MISC'].append(f"{ent.text}({ent.label_})")
        return entities
    
    def classify_sentence(self, text):
        """Classification basée sur les patterns spaCy"""
        doc = self.nlp(normalizer.normalize(text))
        
        # Détection des verbes modaux
        modal_patterns = {
            'obligation': ['يجب', 'يلتزم', 'يتعين'],
            'interdiction': ['يمنع', 'لا يجوز', 'ممنوع'],
            'sanction': ['يعاقب', 'غرامة', 'الحبس'],
        }
        
        for token in doc:
            for cat, patterns in modal_patterns.items():
                if token.text in patterns:
                    return cat
        
        return 'information'


spacy_extractor = SpacyNERExtractor(nlp)

# Test sur un article
test_text = "المادة 183 يعاقب بالحبس من ستة أشهر إلى سنة واحدة كل شخص يسوق مركبة تحت تأثير الكحول"
print(f"Texte: {test_text}")
print(f"Classification spaCy: {spacy_extractor.classify_sentence(test_text)}")
print(f"Entités: {spacy_extractor.extract_entities(test_text)}")

## Classification par clustering avec TF-IDF (Regroupement d'infractions similaires)

In [ ]:
def add_clusters_to_df(df, n_clusters=8):
    """Ajoute une colonne de cluster basée sur les descriptions d'infractions"""
    
    # Préparer les textes
    texts = df['infraction_desc'].fillna('').tolist()
    texts = [normalizer.normalize(t) for t in texts]
    
    # Vectorisation TF-IDF
    vectorizer = TfidfVectorizer(max_features=100, min_df=1, token_pattern=r'[\u0600-\u06FF]{2,}')
    tfidf_matrix = vectorizer.fit_transform(texts)
    
    # Clustering KMeans
    kmeans = KMeans(n_clusters=min(n_clusters, len(texts)), random_state=42, n_init=10)
    clusters = kmeans.fit_predict(tfidf_matrix)
    
    # Ajouter les clusters au DataFrame
    df['cluster'] = clusters
    
    # Afficher les clusters
    feature_names = vectorizer.get_feature_names_out()
    print("\n--- Clusters d'infractions ---")
    for i in range(kmeans.n_clusters):
        cluster_indices = df[df['cluster'] == i].index
        if len(cluster_indices) > 0:
            # Top termes du cluster
            center = kmeans.cluster_centers_[i]
            top_indices = center.argsort()[-5:][::-1]
            top_terms = [feature_names[idx] for idx in top_indices if center[idx] > 0]
            print(f"\nCluster {i}: {len(cluster_indices)} infractions")
            print(f"  Termes clés: {', '.join(top_terms[:5])}")
            # Exemple d'infraction
            sample_idx = cluster_indices[0]
            print(f"  Exemple: {df.iloc[sample_idx]['infraction_desc'][:80]}...")
    
    return df


df = add_clusters_to_df(df, n_clusters=6)

## Export du fichier CSV final

In [ ]:
# Ajouter des colonnes supplémentaires
df['amende_moyenne'] = df.apply(
    lambda r: (r['amende_min'] + r['amende_max']) / 2 if r['amende_min'] and r['amende_max'] else None, 
    axis=1
)

# Réorganisation des colonnes
columns_order = [
    'article_id',
    'infraction_desc',
    'categorie_vehicule',
    'amende_min',
    'amende_max',
    'amende_fixe',
    'amende_moyenne',
    'points_retrait',
    'mots_cles',
    'type_texte',
    'cluster',
    'prison_min',
    'prison_max',
]

df = df[columns_order]

# Export vers CSV
df.to_csv('export_final.csv', index=False, encoding='utf-8-sig')

print("\n" + "="*60)
print("Fichier export_final.csv généré avec succès!")
print(f"Nombre total d'infractions: {len(df)}")
print("="*60)

# Aperçu du résultat
df.head(20)

## Statistiques et validation

In [ ]:
print("\n=== STATISTIQUES ===")
print(f"Articles avec amendes: {df['amende_min'].notna().sum()}")
print(f"Articles avec points: {df['points_retrait'].notna().sum()}")
print(f"\nDistribution par type de texte:")
print(df['type_texte'].value_counts())
print(f"\nDistribution par catégorie de véhicule:")
print(df['categorie_vehicule'].value_counts().head(10))

## Fonction complète pour traiter un fichier PDF

In [ ]:
def process_pdf_to_csv(pdf_path, output_csv='export_final.csv'):
    """
    Pipeline complet: PDF -> Texte -> Parsing -> CSV
    """
    try:
        import fitz  # PyMuPDF
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text("text") + "\n"
        doc.close()
        
        df = parse_articles_to_dataframe(text)
        df = add_clusters_to_df(df)
        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"✅ Traitement terminé! Fichier sauvegardé: {output_csv}")
        return df
    except Exception as e:
        print(f"❌ Erreur: {e}")
        return None


# Exemple d'utilisation (décommentez pour utiliser avec votre PDF)
# df_result = process_pdf_to_csv('data/code_route.pdf', 'code_route_export.csv')

## Résumé final

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                 PIPELINE NLP - CODE DE LA ROUTE                  ║
╠══════════════════════════════════════════════════════════════════╣
║  ✅ Étape A - Prétraitement arabe:                               ║
║     - Suppression Tashkeel (voyelles)                           ║
║     - Normalisation Hamza (أ,إ,آ → ا)                          ║
║     - Suppression Tatweel (ـ)                                   ║
║     - Conversion chiffres arabes → latins                       ║
╠══════════════════════════════════════════════════════════════════╣
║  ✅ Étape B - Extraction par Patterns (Rules-based NLP):        ║
║     - Pattern amendes: (\d+)\s*درهم                           ║
║     - Pattern points: خصم\s+(\d+)\s+نقط                      ║
║     - Pattern catégories: poids_lourd, moto, agricole...       ║
║     - Pattern mots-clés: vitesse, stationnement, alcohol...    ║
╠══════════════════════════════════════════════════════════════════╣
║  ✅ Étape C - ML (spaCy + TF-IDF + Clustering):                ║
║     - spaCy pour NER (entités nommées)                         ║
║     - TF-IDF vectorisation                                     ║
║     - KMeans clustering des infractions similaires             ║
╠══════════════════════════════════════════════════════════════════╣
║  📊 Sortie CSV: export_final.csv                                ║
║  📈 {len(df)} infractions extraites                             ║
║  📋 Colonnes: article_id, infraction_desc, categorie_vehicule, ║
║              amende_min, amende_max, amende_fixe, points_retrait, ║
║              mots_cles, type_texte, cluster                    ║
╚══════════════════════════════════════════════════════════════════╝
""")